In [2]:
import os
import pandas as pd

In [3]:
DATA_DIR = "../data/raw"
SAMPLE_SUB_DIR = "../data"

In [4]:
customers = pd.read_csv(os.path.join(DATA_DIR, "customers.csv"))
geography = pd.read_csv(os.path.join(DATA_DIR, "geography.csv"))
inventory = pd.read_csv(os.path.join(DATA_DIR, "inventory.csv"))
order_items = pd.read_csv(os.path.join(DATA_DIR, "order_items.csv"), dtype={"promo_id_2": str})
orders = pd.read_csv(os.path.join(DATA_DIR, "orders.csv"))
payments = pd.read_csv(os.path.join(DATA_DIR, "payments.csv"))
products = pd.read_csv(os.path.join(DATA_DIR, "products.csv"))
promotions = pd.read_csv(os.path.join(DATA_DIR, "promotions.csv"))
returns = pd.read_csv(os.path.join(DATA_DIR, "returns.csv"))
reviews = pd.read_csv(os.path.join(DATA_DIR, "reviews.csv"))
sales = pd.read_csv(os.path.join(DATA_DIR, "sales.csv"))
shipments = pd.read_csv(os.path.join(DATA_DIR, "shipments.csv"))
web_traffic = pd.read_csv(os.path.join(DATA_DIR, "web_traffic.csv"))

sample_sub = pd.read_csv(os.path.join(SAMPLE_SUB_DIR, "sample_submission.csv"))

**Q1.** Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ `orders.csv`)

In [5]:
orders.head()

,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source
0,1,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search
1,2,2012-07-04,58621,1330,returned,cod,mobile,paid_search
2,3,2012-07-04,58811,1473,delivered,credit_card,desktop,direct
3,4,2012-07-04,59453,2360,delivered,credit_card,desktop,referral
4,6,2012-07-06,57821,2886,delivered,paypal,mobile,email_campaign


In [ ]:
# 1. Chuyển cột order_date sang định dạng datetime
orders['order_date'] = pd.to_datetime(orders['order_date'])

# 2. Lọc các đơn hàng "delivered" (được coi là lần mua thành công)
# Theo logic kinh doanh, "lần mua" thường không bao gồm các đơn bị hủy (cancelled)
purchases = orders[orders['order_status'] == 'delivered'].copy()


# 3. Sắp xếp dữ liệu theo customer_id và order_date
purchases = purchases.sort_values(['customer_id', 'order_date'])

# 4. Tính số ngày chênh lệch giữa các đơn hàng liên tiếp của cùng một khách hàng
purchases['inter_order_gap'] = purchases.groupby('customer_id')['order_date'].diff().dt.days

# 5. Tính trung vị (median) của các khoảng cách này
median_gap = purchases['inter_order_gap'].dropna().median()

print(f"Trung vị số ngày giữa hai lần mua liên tiếp là: {median_gap} ngày")

In [10]:
purchases.head()

,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source,inter_order_gap
4023,5280,2012-07-25,1,15201,delivered,cod,desktop,paid_search,NaN
238890,308113,2015-07-31,1,15201,delivered,cod,mobile,paid_search,1101.0
374571,483190,2017-04-23,1,15201,delivered,cod,mobile,paid_search,632.0
544446,702081,2020-02-24,1,15201,delivered,credit_card,mobile,organic_search,1037.0
91192,117673,2013-09-20,2,15201,delivered,paypal,mobile,social_media,NaN


**=> Đáp án C**

**Q2.** Phân khúc sản phẩm (segment) nào trong `products.csv` có tỷ suất lợi nhuận gộp
trung bình cao nhất, với công thức (price − cogs)/price?

In [16]:
products.head()

,product_id,product_name,category,segment,size,color,price,cogs
0,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.650000,9704.842875
1,537,SaigonFlex UC-02,Streetwear,Everyday,M,silver,9523.076013,5393.870254
2,538,SaigonFlex UC-03,Streetwear,Everyday,L,pink,15951.633158,11371.919278
3,539,SaigonFlex UC-04,Streetwear,Everyday,XL,yellow,15753.717299,8573.172954
4,540,SaigonFlex UC-05,Streetwear,Everyday,S,red,15766.334536,14063.570406


In [19]:
# 1. Tính tỷ suất lợi nhuận gộp cho từng sản phẩm
# Công thức: (giá bán - giá vốn) / giá bán
products['gross_margin'] = (products['price'] - products['cogs']) / products['price']

# 2. Tính trung bình tỷ suất lợi nhuận gộp theo từng phân khúc (segment)
segment_analysis = products.groupby('segment')['gross_margin'].mean().sort_values(ascending=False)

# 3. Hiển thị kết quả
print("Tỷ suất lợi nhuận gộp trung bình theo phân khúc:")
print(segment_analysis)

Tỷ suất lợi nhuận gộp trung bình theo phân khúc:
segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343
Name: gross_margin, dtype: float64


**=> Đáp án D**

**Q3.** Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (`join
returns với products theo product_id`), lý do trả hàng nào xuất hiện nhiều nhất?

In [20]:
# 1. Lọc danh sách các sản phẩm thuộc danh mục 'Streetwear'
streetwear_products = products[products['category'] == 'Streetwear']

# 2. Kết nối bảng trả hàng với danh sách sản phẩm Streetwear theo product_id
# Sử dụng inner join để chỉ giữ lại các bản ghi trả hàng của Streetwear
streetwear_returns = returns.merge(streetwear_products, on='product_id')

# 3. Thống kê tần suất xuất hiện của các lý do trả hàng
reason_counts = streetwear_returns['return_reason'].value_counts()

# 4. Hiển thị kết quả
print("Thống kê lý do trả hàng của danh mục Streetwear:")
print(reason_counts)

Thống kê lý do trả hàng của danh mục Streetwear:
return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64


**=> Đáp án B**

**Q4.** Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung
bình (bounce_rate) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?

In [22]:
web_traffic.head()

,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
0,2013-01-01,9760,7253,39093,0.00514,102.9,organic_search
1,2013-01-02,10456,8151,47611,0.00406,120.5,organic_search
2,2013-01-03,10076,7458,36963,0.00401,263.6,direct
3,2013-01-04,9973,8063,53078,0.00562,151.8,direct
4,2013-01-05,10223,7882,36790,0.00525,168.6,referral


In [21]:
# 2. Tính tỷ lệ thoát trung bình cho từng nguồn truy cập (traffic_source)
avg_bounce_rates = web_traffic.groupby('traffic_source')['bounce_rate'].mean().sort_values()

# 3. Hiển thị kết quả
print("Tỷ lệ thoát trung bình theo nguồn truy cập:")
print(avg_bounce_rates)

Tỷ lệ thoát trung bình theo nguồn truy cập:
traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: float64


**=> Đáp án C**

**Q5.** Tỷ lệ phần trăm các dòng trong `order_items.csv` có áp dụng khuyến mãi (tức là promo_id
không null) xấp xỉ là bao nhiêu?

In [23]:
# 2. Tính tổng số dòng trong bảng
total_rows = len(order_items)

# 3. Đếm số dòng mà promo_id không phải là giá trị null (NaN)
non_null_promo_count = order_items['promo_id'].notna().sum()

# 4. Tính tỷ lệ phần trăm
percentage = (non_null_promo_count / total_rows) * 100

print(f"Tổng số dòng: {total_rows}")
print(f"Số dòng có khuyến mãi: {non_null_promo_count}")
print(f"Tỷ lệ phần trăm: {percentage:.2f}%")

Tổng số dòng: 714669
Số dòng có khuyến mãi: 276316
Tỷ lệ phần trăm: 38.66%


**=> Đáp án C**

**Q6.** Trong `customers.csv`, xét các khách hàng có age_group khác null, nhóm tuổi nào có số
đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong
nhóm)

In [24]:
# 2. Lọc khách hàng có age_group khác null
customers_filtered = customers[customers['age_group'].notna()]

# 3. Đếm tổng số khách hàng trong mỗi nhóm tuổi
customer_counts = customers_filtered.groupby('age_group')['customer_id'].nunique()

# 4. Kết nối bảng orders với bảng khách hàng đã lọc để lấy thông tin nhóm tuổi cho mỗi đơn hàng
order_with_age = orders.merge(customers_filtered[['customer_id', 'age_group']], on='customer_id')

# 5. Tính tổng số đơn hàng cho mỗi nhóm tuổi
order_counts = order_with_age.groupby('age_group')['order_id'].count()

# 6. Tính số đơn hàng trung bình trên mỗi khách hàng (Tổng đơn / Tổng khách)
avg_orders_per_customer = (order_counts / customer_counts).sort_values(ascending=False)

# 7. Hiển thị kết quả
print("Số đơn hàng trung bình trên mỗi khách hàng theo nhóm tuổi:")
print(avg_orders_per_customer)

Số đơn hàng trung bình trên mỗi khách hàng theo nhóm tuổi:
age_group
55+      5.406851
45-54    5.357241
35-44    5.337343
25-34    5.245226
18-24    5.226656
dtype: float64


**=> Đáp án A**

**Q7.** Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất trong
sales_train.csv?

In [26]:
sales.tail()

,Date,Revenue,COGS
3828,2022-12-27,2100553.66,2184872.24
3829,2022-12-28,3448729.20,3513621.00
3830,2022-12-29,3083944.33,3170787.10
3831,2022-12-30,2884668.76,3022292.15
3832,2022-12-31,2383037.48,2279288.13


In [27]:
# 2. Tính doanh thu cho từng dòng trong order_items
# Công thức: (số lượng * đơn giá) - số tiền giảm giá
order_items['revenue'] = (order_items['quantity'] * order_items['unit_price']) - order_items['discount_amount']

# 3. Tính tổng doanh thu theo từng mã đơn hàng (order_id)
order_revenue = order_items.groupby('order_id')['revenue'].sum().reset_index()

# 4. Kết nối với bảng orders để lấy mã zip của mỗi đơn hàng
# Lưu ý: Thông thường doanh thu thực tế được tính trên các đơn "delivered"
order_geo = order_revenue.merge(orders[['order_id', 'zip', 'order_status']], on='order_id')

# 5. Kết nối với bảng geography để lấy thông tin vùng (region) theo mã zip
revenue_by_region = order_geo.merge(geography[['zip', 'region']], on='zip')

# 6. Tổng hợp doanh thu theo vùng
total_revenue_region = revenue_by_region.groupby('region')['revenue'].sum().sort_values(ascending=False)

print("Tổng doanh thu theo vùng miền:")
print(total_revenue_region)

Tổng doanh thu theo vùng miền:
region
East       7.291151e+09
Central    4.719491e+09
West       3.670227e+09
Name: revenue, dtype: float64


**=> Đáp án C**

**Q8.** Trong các đơn hàng có order_status = ’cancelled’ trong `orders.csv`, phương thức
thanh toán nào được sử dụng nhiều nhất?

In [28]:
# 2. Lọc các đơn hàng có trạng thái là 'cancelled'
cancelled_orders = orders[orders['order_status'] == 'cancelled']

# 3. Thống kê số lượng theo từng phương thức thanh toán
payment_stats = cancelled_orders['payment_method'].value_counts()

# 4. Hiển thị kết quả
print("Thống kê phương thức thanh toán của các đơn hàng bị hủy:")
print(payment_stats)

Thống kê phương thức thanh toán của các đơn hàng bị hủy:
payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64


**=> Đáp án A**

**Q9.** Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao
nhất, được định nghĩa là số bản ghi trong returns chia cho số dòng trong order_items (join
với products theo product_id)?

In [29]:
# 2. Gán thông tin kích cỡ (size) cho từng dòng trong chi tiết đơn hàng
order_items_with_size = order_items.merge(products[['product_id', 'size']], on='product_id')
# Đếm tổng số dòng order_items theo từng kích cỡ
order_counts = order_items_with_size['size'].value_counts()

# 3. Gán thông tin kích cỡ (size) cho từng bản ghi trong bảng trả hàng
returns_with_size = returns.merge(products[['product_id', 'size']], on='product_id')
# Đếm tổng số bản ghi trả hàng theo từng kích cỡ
return_counts = returns_with_size['size'].value_counts()

# 4. Tập trung vào 4 kích thước mục tiêu: S, M, L, XL
target_sizes = ['S', 'M', 'L', 'XL']
order_counts = order_counts.reindex(target_sizes)
return_counts = return_counts.reindex(target_sizes)

# 5. Tính tỷ lệ trả hàng (số bản ghi trả hàng / số dòng chi tiết đơn hàng)
return_rates = (return_counts / order_counts).sort_values(ascending=False)

# 6. Hiển thị kết quả
print("Tỷ lệ trả hàng theo kích thước:")
print(return_rates)

Tỷ lệ trả hàng theo kích thước:
size
S     0.056515
L     0.056250
M     0.055660
XL    0.055200
Name: count, dtype: float64


**=> Đáp án A**

**Q10.** Trong `payments.csv`, kế hoạch trả góp nào có giá trị thanh toán trung bình trên
mỗi đơn hàng cao nhất?

In [30]:
# 2. Tính giá trị thanh toán trung bình cho mỗi loại kỳ hạn trả góp (installments)
avg_payment_by_plan = payments.groupby('installments')['payment_value'].mean().sort_values(ascending=False)

# 3. Hiển thị kết quả
print("Giá trị thanh toán trung bình theo kế hoạch trả góp:")
print(avg_payment_by_plan)

Giá trị thanh toán trung bình theo kế hoạch trả góp:
installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
2       708.473729
Name: payment_value, dtype: float64


**=> Đáp án C**